# Re-encode Note Embeddings with ClinicalBERT (post admission-med fix)

The `notes_text_mimic3.pkl` was re-extracted locally with the updated `extract_notes.py`
which now removes **admission/home/current medication sections** in addition to discharge meds.
This notebook re-encodes those cleaned texts into `note_embeddings_mimic3.pkl`.

**Inputs (upload to Kaggle dataset):**
- `notes_text_mimic3.pkl` — cleaned note texts (regenerated locally)
- `cohort_mimic3.pkl` — cohort hadm_ids

**Output:** `note_embeddings_mimic3.pkl` (download and replace in `data/processed/`)

~20–30 min on Kaggle P100.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q transformers

In [ ]:
import os, glob, shutil

os.makedirs("data/processed", exist_ok=True)

# Find uploaded files in /kaggle/input
for fname in ["notes_text_mimic3.pkl", "cohort_mimic3.pkl"]:
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{fname} not found in /kaggle/input — upload it to the dataset")
    shutil.copy(hits[0], f"data/processed/{fname}")
    print(f"Copied {hits[0]} → data/processed/{fname}")

In [ ]:
import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load cohort and note texts
with open("data/processed/cohort_mimic3.pkl", "rb") as f:
    cohort = pickle.load(f)
hadm_ids = np.array(cohort["hadm_ids"])
print(f"Cohort: {len(hadm_ids):,} admissions")

with open("data/processed/notes_text_mimic3.pkl", "rb") as f:
    notes_data = pickle.load(f)
note_lookup = notes_data["notes"]  # dict: hadm_id -> cleaned text
print(f"Notes loaded: {len(note_lookup):,} texts")

# Load model
model_name = "emilyalsentzer/Bio_ClinicalBERT"
print(f"Loading {model_name} ...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()
print("Model loaded")

MAX_LENGTH = 512
OVERLAP = 128
BATCH_SIZE = 32
STRIDE = MAX_LENGTH - OVERLAP

def chunk_text(text):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for start in range(0, len(tokens), STRIDE):
        chunk = tokens[start:start + MAX_LENGTH]
        if len(chunk) < 32:
            break
        chunks.append(chunk)
    if not chunks and tokens:
        chunks.append(tokens[:MAX_LENGTH])
    return chunks

n = len(hadm_ids)
embeddings = np.zeros((n, 768), dtype=np.float32)
has_note = np.zeros(n, dtype=np.float32)
hadm_to_idx = {h: i for i, h in enumerate(hadm_ids)}

processed = 0
for hadm_id in hadm_ids:
    text = note_lookup.get(int(hadm_id))
    if not text or len(text.strip()) < 50:
        continue

    idx = hadm_to_idx[hadm_id]
    chunks = chunk_text(text)
    if not chunks:
        continue

    chunk_embeds = []
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        max_len = max(len(c) for c in batch)
        input_ids = [c + [tokenizer.pad_token_id] * (max_len - len(c)) for c in batch]
        attn_mask = [[1] * len(c) + [0] * (max_len - len(c)) for c in batch]

        ids_t  = torch.tensor(input_ids, dtype=torch.long, device=device)
        mask_t = torch.tensor(attn_mask, dtype=torch.long, device=device)

        with torch.no_grad():
            out = model(input_ids=ids_t, attention_mask=mask_t)
            hidden = out.last_hidden_state  # (B, seq, 768)
            m_exp  = mask_t.unsqueeze(-1).float()
            mean_e = (hidden * m_exp).sum(1) / m_exp.sum(1).clamp(min=1)
            chunk_embeds.append(mean_e.cpu().numpy())

    all_c = np.concatenate(chunk_embeds, axis=0)
    if all_c.shape[0] == 1:
        embeddings[idx] = all_c[0]
    else:
        norms = np.linalg.norm(all_c, axis=1)
        w = norms / (norms.sum() + 1e-8)
        embeddings[idx] = (w[:, None] * all_c).sum(0)
    has_note[idx] = 1.0

    processed += 1
    if processed % 1000 == 0:
        print(f"  {processed:,}/{n:,} encoded ...")

print(f"Done: {processed:,}/{n:,} admissions with notes")
mean_norm = np.linalg.norm(embeddings[has_note == 1], axis=1).mean()
print(f"Mean embedding L2 norm: {mean_norm:.2f}")

out = {
    "embeddings": embeddings,
    "has_note":   has_note,
    "hadm_ids":   hadm_ids,
    "method":     "clinicalbert_chunk_pool",
}
with open("note_embeddings_mimic3.pkl", "wb") as f:
    pickle.dump(out, f)
print("Saved → note_embeddings_mimic3.pkl  (download this file)")